In [ ]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=FFVe99H1UlU1LU2nj2QUeNAHsM7dUJ&access_type=offline&code_challenge=uHEGaG5JEbp5TIR-fsVwvuJ8MRbMFCU33uZjVEDfYCE&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [ ]:
!gcloud auth login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=UiDTZiS2COSkwLlSyy0l1pVKoF8Yfg&access_type=offline&code_challenge=4jMAWGLzSEe-ozTnSiVWYKCSv6hg9PyZzX727eiHbYk&code_challenge_method=S256


You are now logged in as [yt4@sanger.ac.uk].
Your current project is [open-targets-genetics-dev].  You can change this setting by running:
  $ gcloud config set project PROJECT_ID


Updates are available for some Google Cloud CLI components.  To install them,
please run:
  $ gcloud components update



In [1]:
import os

import hail as hl
import numpy as np
import pyspark.sql.functions as f
from pyspark.sql import DataFrame

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.dataset.study_locus import StudyLocus
from gentropy.susie_finemapper import SusieFineMapperStep
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment

"""Common utilities for the project."""

import os
from pathlib import Path
from gentropy.common.session import Session
import logging


def get_gcs_credentials() -> str:
    """Get the credentials for google cloud storage."""
    app_default_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/application_default_credentials.json"
    )

    service_account_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/service_account_credentials.json"
    )

    if Path(app_default_credentials).exists():
        return app_default_credentials
    else:
        raise FileNotFoundError("No GCS credentials found.")


def get_gcs_hadoop_connector_jar() -> str:
    """Get the google cloud storage hadoop connector for spark.

    This function will return the url to download the hadoop jar.
    """

    return (
        "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"
    )


def gcs_conf(
    credentials_path=None, project="open-targets-genetics-dev"
) -> dict[str, str]:
    """Get the spark configuration with hadoop connector for google cloud storage."""
    credentials_path = credentials_path or get_gcs_credentials()
    return {
        "spark.driver.memory": "12g",
        "spark.kryoserializer.buffer.max": "500m",
        "spark.driver.maxResultSize":"2g",
        "spark.hadoop.fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        "spark.jars": get_gcs_hadoop_connector_jar(),
        "spark.hadoop.google.cloud.auth.service.account.enable": "true",
        "spark.hadoop.fs.gs.project.id": project,
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile": credentials_path,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
    }


class GentropySession(Session):
    def __init__(self, *args, **kwargs):
        if "extended_spark_conf" in kwargs:
            kwargs["extended_spark_conf"].update(gcs_conf())
        else:
            kwargs["extended_spark_conf"] = gcs_conf()
        super().__init__(*args, **kwargs)

    @property
    def conf(self):
        logging.warning(
            "To change the config restart the session and use the `extended_spark_conf` parameter."
        )
        return self.spark.sparkContext.getConf().getAll()

session= GentropySession()


path_to_release_folder="gs://open-targets-data-releases/25.03/"
#path_to_release_folder="gs://open-targets-pre-data-releases/24.12-uo_test-3/output/genetics/parquet/"
#path_to_release_folder="gs://ot_orchestration/releases/25.02_freeze1/"

si=StudyIndex.from_parquet(session,path_to_release_folder+"output/study/")
sl=StudyLocus.from_parquet(session,path_to_release_folder+"output/credible_set/")

Loading BokehJS ...

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/12 17:29:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/05/12 17:29:13 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


# The list of qulifying studies

In [2]:
df=session.spark.read.parquet("gs://genetics-portal-dev-analysis/ss60/gentropy-manuscript/chapters/variant-effect-prediction/rescaled-betas.parquet").cache()

In [3]:
df.count()

2621048

In [7]:
df.printSchema()

root
 |-- variantId: string (nullable = true)
 |-- studyId: string (nullable = true)
 |-- studyLocusId: string (nullable = true)
 |-- beta: double (nullable = true)
 |-- zScore: double (nullable = true)
 |-- pValueMantissa: float (nullable = true)
 |-- pValueExponent: integer (nullable = true)
 |-- standardError: double (nullable = true)
 |-- finemappingMethod: string (nullable = true)
 |-- studyType: string (nullable = true)
 |-- credibleSetSize: integer (nullable = true)
 |-- posteriorProbability: double (nullable = true)
 |-- nSamples: integer (nullable = true)
 |-- nControls: integer (nullable = true)
 |-- nCases: integer (nullable = true)
 |-- majorPopulation: struct (nullable = true)
 |    |-- ldPopulation: string (nullable = true)
 |    |-- relativeSampleSize: double (nullable = true)
 |-- allelefrequencies: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- populationName: string (nullable = true)
 |    |    |-- alleleFrequency: double (nulla

In [4]:
ta=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/gwas_study_index_with_theraputic_areas").cache()

25/04/25 14:42:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [5]:
ta.count()

96404

In [6]:
ta.show()

+--------------------+-----------+---------+--------------------+------------------------+--------------------+------+---------------------+-----------+--------+--------------------+----------------------+---------------+------------------+----------------------------------+--------------------+--------------------+------+---------+--------+---------+---------------------+--------------------+--------------------+--------------------+-------------+--------------------+-----------+---------+--------------------+-----------+---------+----------+--------------+---------+------+----------+-----------+-------------+--------------+--------+-----------+---------+---------+---------------+---------+-----------+-----------------+-----+
|             studyId|  projectId|studyType|     traitFromSource|traitFromSourceMappedIds|          diseaseIds|geneId|biosampleFromSourceId|biosampleId|pubmedId|    publicationTitle|publicationFirstAuthor|publicationDate|publicationJournal|backgroundTraitFromSour

In [8]:
columns_ta = [
    "Haematology", "Metabolic", "Congenital", "Signs/symptoms", "Neurology",
    "Immune", "Psychiatry", "Dermatology", "Ophthalmology", "Cardiovascular",
    "Respiratory", "Digestive", "Endocrine", "Musculoskeletal", "Infection","Oncology",
    "Other"
]

In [9]:
ta = ta.withColumn(
    "sum_columns_ta",
    sum(f.col(column) for column in columns_ta)
).cache()
ta.show()

+--------------------+-----------+---------+--------------------+------------------------+--------------------+------+---------------------+-----------+--------+--------------------+----------------------+---------------+------------------+----------------------------------+--------------------+--------------------+------+---------+--------+---------+---------------------+--------------------+--------------------+--------------------+-------------+--------------------+-----------+---------+--------------------+-----------+---------+----------+--------------+---------+------+----------+-----------+-------------+--------------+--------+-----------+---------+---------+---------------+---------+-----------+-----------------+-----+--------------+
|             studyId|  projectId|studyType|     traitFromSource|traitFromSourceMappedIds|          diseaseIds|geneId|biosampleFromSourceId|biosampleId|pubmedId|    publicationTitle|publicationFirstAuthor|publicationDate|publicationJournal|backgrou

In [10]:
columns_ta_no_oncology = [
    "Haematology", "Metabolic", "Congenital", "Signs/symptoms", "Neurology",
    "Immune", "Psychiatry", "Dermatology", "Ophthalmology", "Cardiovascular",
    "Respiratory", "Digestive", "Endocrine", "Musculoskeletal", "Infection",
    "Other"
]

In [11]:
ta = ta.withColumn(
    "sum_columns_ta_no_oncology",
    sum(f.col(column) for column in columns_ta_no_oncology)
).cache()
ta.show()

+--------------------+-----------+---------+--------------------+------------------------+--------------------+------+---------------------+-----------+--------+--------------------+----------------------+---------------+------------------+----------------------------------+--------------------+--------------------+------+---------+--------+---------+---------------------+--------------------+--------------------+--------------------+-------------+--------------------+-----------+---------+--------------------+-----------+---------+----------+--------------+---------+------+----------+-----------+-------------+--------------+--------+-----------+---------+---------+---------------+---------+-----------+-----------------+-----+--------------+--------------------------+
|             studyId|  projectId|studyType|     traitFromSource|traitFromSourceMappedIds|          diseaseIds|geneId|biosampleFromSourceId|biosampleId|pubmedId|    publicationTitle|publicationFirstAuthor|publicationDate|

In [ ]:
ta = ta.withColumn(
    "sum_columns_ta_no_oncology",
    sum(f.col(column) for column in columns_ta_no_oncology)
).cache()
ta.show()

In [18]:
ta = ta.withColumn(
    "prev",
    f.col("nCases")/f.col("nSamples")
).cache()
ta.show()

+--------------------+-----------+---------+--------------------+------------------------+--------------------+------+---------------------+-----------+--------+--------------------+----------------------+---------------+------------------+----------------------------------+--------------------+--------------------+------+---------+--------+---------+---------------------+--------------------+--------------------+--------------------+-------------+--------------------+-----------+---------+--------------------+-----------+---------+----------+--------------+---------+------+----------+-----------+-------------+--------------+--------+-----------+---------+---------+---------------+---------+-----------+-----------------+-----+--------------+--------------------------+--------------------+
|             studyId|  projectId|studyType|     traitFromSource|traitFromSourceMappedIds|          diseaseIds|geneId|biosampleFromSourceId|biosampleId|pubmedId|    publicationTitle|publicationFirstAu

In [19]:
n_theshold=10000
prev_threshold=0.005

In [20]:
# Qulified studies #1
qualified_studies = ta.filter(
    (f.col("sum_columns_ta") > 0) & (f.col("nSamples") > n_theshold) & (f.col("prev") > prev_threshold)
).select("studyId").distinct().cache()
qualified_studies.count()

7300

In [21]:
qualified_studies.write.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/qualified_studies_with_oncology",mode="overwrite")

In [22]:
# Qulified studies #2
qualified_studies = ta.filter(
    (f.col("sum_columns_ta_no_oncology") > 0) & (f.col("nSamples") > n_theshold) & (f.col("prev") > prev_threshold)
).select("studyId").distinct().cache()
qualified_studies.count()

6282

In [23]:
qualified_studies.write.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/qualified_studies_without_oncology",mode="overwrite")

# Qulified measurments

In [7]:
ta=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/gwas_study_index_with_theraputic_areas").cache()

25/05/12 17:32:21 WARN CacheManager: Asked to cache already cached data.


In [8]:
ta=ta.filter((f.col("Measurement")==1) & (f.col("bianry_less_cases")==0)).cache()
ta.count()

25/05/12 17:32:22 WARN CacheManager: Asked to cache already cached data.


74088

In [ ]:
#EFO_0007882 - microbiome
#EFO_0004747 - protein measurement

In [9]:
disease_index_path=path_to_release_folder+"output/disease/disease.parquet"
disease_index_orig = session.spark.read.parquet(disease_index_path)

In [10]:
efo_to_assign=chemblDrugEnrichment.selecting_all_decendands_based_on_efo_list(
        disease_index_orig=disease_index_orig, efo_ids=["EFO_0007882","EFO_0004747"])
len(efo_to_assign)

3686

In [11]:
efo_to_assign

['EFO_0008167',
 'EFO_0008181',
 'EFO_0010586',
 'EFO_0020110',
 'EFO_0020628',
 'EFO_0020658',
 'EFO_0020661',
 'EFO_0020663',
 'EFO_0020759',
 'EFO_0020780',
 'EFO_0020838',
 'EFO_0020905',
 'EFO_0021775',
 'EFO_0801477',
 'EFO_0801785',
 'EFO_0801872',
 'EFO_0801899',
 'EFO_0802286',
 'EFO_0802318',
 'EFO_0802521',
 'EFO_0802738',
 'EFO_0802768',
 'EFO_0802794',
 'EFO_0802807',
 'EFO_0803037',
 'EFO_0803101',
 'EFO_0803107',
 'EFO_0803310',
 'EFO_0005845',
 'EFO_0008132',
 'EFO_0008190',
 'EFO_0008463',
 'EFO_0020751',
 'EFO_0021870',
 'EFO_0022173',
 'EFO_0022285',
 'EFO_0801618',
 'EFO_0801730',
 'EFO_0802160',
 'EFO_0802215',
 'EFO_0802254',
 'EFO_0802564',
 'EFO_0802804',
 'EFO_0802822',
 'EFO_0005000',
 'EFO_0008114',
 'EFO_0020596',
 'EFO_0020635',
 'EFO_0020644',
 'EFO_0020840',
 'EFO_0020953',
 'EFO_0022272',
 'EFO_0801346',
 'EFO_0801413',
 'EFO_0801615',
 'EFO_0801897',
 'EFO_0801926',
 'EFO_0801960',
 'EFO_0801985',
 'EFO_0802276',
 'EFO_0802344',
 'EFO_0802580',
 'EFO_08

In [12]:
ta.show(1)

+------------+---------+---------+-------------------+------------------------+-------------+------+---------------------+-----------+--------+--------------------+----------------------+---------------+------------------+----------------------------------+--------------------+--------------------+------+---------+--------+-------+---------------------+------------------+------------------+--------------------+-------------+--------------------+-----------+---------+---------------+-----------+---------+----------+--------------+---------+------+----------+-----------+-------------+--------------+--------+-----------+---------+---------+---------------+---------+-----------+-----------------+-----+
|     studyId|projectId|studyType|    traitFromSource|traitFromSourceMappedIds|   diseaseIds|geneId|biosampleFromSourceId|biosampleId|pubmedId|    publicationTitle|publicationFirstAuthor|publicationDate|publicationJournal|backgroundTraitFromSourceMappedIds|backgroundDiseaseIds|   initialSamp

In [15]:
from pyspark.sql.functions import explode, col

# Explode the diseaseIds array into individual rows
exploded_ta = ta.withColumn("diseaseId", explode(col("diseaseIds")))

# Show the result
exploded_ta.count()

78408

In [16]:
sids_to_excl=exploded_ta.filter(f.col("diseaseId").isin(efo_to_assign)).select("studyId").distinct().cache()
sids_to_excl.count()

13637

In [17]:
ta=ta.join(sids_to_excl, on="studyId", how="left_anti").cache()
ta.count()

60451

In [18]:
ta=ta.select("studyId")
ta.count()

60451

In [19]:
ta.write.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/qualified_studies_measurements",mode="overwrite")

# Qulified credible sets

In [26]:
# Qulified studies #1
qualified_studies = ta.filter(
    (f.col("sum_columns_ta") > 0) & (f.col("nSamples") > n_theshold) & (f.col("prev") > prev_threshold)
).select("studyId").distinct().cache()
qualified_studies.count()

25/04/25 16:06:54 WARN CacheManager: Asked to cache already cached data.


7300

In [30]:
df_1=df.join(qualified_studies,on="studyId",how="inner").cache()
df_1.count()

25/04/25 16:09:46 WARN CacheManager: Asked to cache already cached data.


52035

In [31]:
df_1=df_1.filter(
    (f.col("rescaledStatistics.estimatedBeta")<=3) &
    (f.col("rescaledStatistics.estimatedBeta")>=-3)&
    (f.col("majorPopulationMAF")>0)
).select("studyLocusId").distinct().cache()
df_1.count()

51345

In [32]:
df_1.write.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/qualified_CSs_with_oncology",mode="overwrite")

In [33]:
# Qulified studies #2
qualified_studies = ta.filter(
    (f.col("sum_columns_ta_no_oncology") > 0) & (f.col("nSamples") > n_theshold) & (f.col("prev") > prev_threshold)
).select("studyId").distinct().cache()
qualified_studies.count()

25/04/25 16:11:36 WARN CacheManager: Asked to cache already cached data.


6282

In [34]:
df_1=df.join(qualified_studies,on="studyId",how="inner").cache()
df_1.count()

43869

In [35]:
df_1=df_1.filter(
    (f.col("rescaledStatistics.estimatedBeta")<=3) &
    (f.col("rescaledStatistics.estimatedBeta")>=-3)&
    (f.col("majorPopulationMAF")>0)
).select("studyLocusId").distinct().cache()
df_1.count()

43276

In [36]:
df_1.write.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/qualified_CSs_without_oncology",mode="overwrite")